# 01 - Ticketmaster API Exploration

## Goal

Explore the Ticketmaster Discovery API and understand the event data
available for building GigRoute Europe.

### Questions

- How are events returned?
- What artist information is available?
- Are venue coordinates available?
- How are cities and countries represented?
- How do pagination and filters work?
- What missing or inconsistent data do we encounter?

In [ ]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

In [ ]:
project_path = Path("..")
load_dotenv(project_path/ ".env")

api_key = os.getenv("TICKETMASTER_API_KEY")

In [ ]:
project_path = Path("..")

load_dotenv(project_path / ".env", override=True)

api_key = os.getenv("TICKETMASTER_API_KEY")

assert api_key is not None, "Ticketmaster API key is missing"

In [ ]:
url = "https://app.ticketmaster.com/discovery/v2/events.json"

params = {
    "classificationName": "music",
    "countryCode": "DE",
    "size": 20
}

In [ ]:
response = requests.get(
    url,
    params={
        **params,
        "apikey": api_key
    },
    timeout=30
)

if not response.ok:
    raise RuntimeError(
        f"Ticketmaster API request failed with status {response.status_code}"
    )

data = response.json()

In [ ]:
print("Top-level keys:", data.keys())
print("Pagination:", data.get("page"))

events = data.get("_embedded", {}).get("events", [])

print("Events returned:", len(events))

## JSON Structure

Ticketmaster returns nested JSON data.

- `{}` represents a dictionary
- `[]` represents a list
- Event details are stored at the top level
- Venue and artist information are nested under `_embedded`
- Missing fields are accessed safely using `.get()`

In [ ]:
rows = []

for event in all_events:
    embedded = event.get("_embedded", {})

    venues = embedded.get("venues", [])
    venue = venues[0] if venues else {}

    attractions = embedded.get("attractions", [])
    artist = attractions[0] if attractions else {}

    location = venue.get("location", {})

    rows.append({
        "event_id": event.get("id"),
        "event_name": event.get("name"),
        "artist_name": artist.get("name"),
        "event_date": event.get("dates", {}).get("start", {}).get("localDate"),
        "event_time": event.get("dates", {}).get("start", {}).get("localTime"),
        "venue_name": venue.get("name"),
        "city": venue.get("city", {}).get("name"),
        "country": venue.get("country", {}).get("name"),
        "latitude": location.get("latitude"),
        "longitude": location.get("longitude"),
        "event_url": event.get("url")
    })

events_df = pd.DataFrame(rows)
events_df.head()

In [ ]:
events_df.shape

In [ ]:
# 2
events_df[
    events_df["event_id"].duplicated(keep=False)
].sort_values("event_id").head(20)

In [ ]:
# 3
events_df[
    events_df["venue_name"].isna()
].head()

## Paginated Event Collection

The Ticketmaster API paginates search results and limits deep paging for broad queries.

To collect a larger set of events safely, requests are split into monthly date ranges and each date range is paginated separately.

In [ ]:
from datetime import datetime, timezone

start_date = pd.Timestamp.now(tz="UTC").normalize()
end_date = start_date + pd.DateOffset(months=12)

date_ranges = pd.date_range(
    start=start_date,
    end=end_date,
    freq="MS"
)

date_ranges

In [ ]:
monthly_windows = []

current = start_date

while current < end_date:
    next_month = current + pd.DateOffset(months=1)

    window_end = min(next_month, end_date)

    monthly_windows.append(
        (current, window_end)
    )

    current = next_month

In [ ]:
def to_ticketmaster_datetime(timestamp):
    return timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
all_events = []

for window_start, window_end in monthly_windows:

    base_params = {
        "classificationName": "music",
        "countryCode": "DE",
        "startDateTime": to_ticketmaster_datetime(window_start),
        "endDateTime": to_ticketmaster_datetime(window_end),
        "size": 20
    }

    # First request for this date window
    response = requests.get(
        url,
        params={
            **base_params,
            "page": 0,
            "apikey": api_key
        },
        timeout=30
    )

    if not response.ok:
        print(
            f"Skipped {window_start.date()} → {window_end.date()} "
            f"| Status: {response.status_code}"
        )
        continue

    page_data = response.json()

    total_pages = page_data.get("page", {}).get("totalPages", 0)
    total_elements = page_data.get("page", {}).get("totalElements", 0)

    print(
        f"\n{window_start.date()} → {window_end.date()} "
        f"| Events available: {total_elements}"
    )

    for page in range(total_pages):

        response = requests.get(
            url,
            params={
                **base_params,
                "page": page,
                "apikey": api_key
            },
            timeout=30
        )

        if not response.ok:
            print(
                f"  Page {page} failed "
                f"| Status: {response.status_code}"
            )
            break

        page_data = response.json()

        page_events = (
            page_data
            .get("_embedded", {})
            .get("events", [])
        )

        all_events.extend(page_events)

    print(
        f"Total collected so far: {len(all_events)}"
    )

In [ ]:
len(all_events)

In [ ]:
all_events[0].get("name")

In [95]:
import json

raw_path = project_path / "data" / "raw" / "ticketmaster"
raw_path.mkdir(parents=True, exist_ok=True)

with open(raw_path / "germany_music_events.json", "w", encoding="utf-8") as file:
    json.dump(all_events, file, ensure_ascii=False, indent=2)

## Initial Data Quality Findings

The initial API exploration identified several data-quality issues:

- Duplicate event IDs are present.
- Some events are missing venue names.
- A small number of events are missing artist or event-time information.
- Geographic fields such as city, country, latitude, and longitude are largely available.
- Further investigation and cleaning will be performed in the next notebook.